In [0]:
from pyspark import pipelines as dp
from trip_transforms import prepare_bronze


trip_volume = spark.conf.get("nyc_taxi.trip_volume")

yellow_source_path = f"{trip_volume}/yellow"
green_source_path = f"{trip_volume}/green"
zones_source_path = f"{trip_volume}/reference/taxi_zone_lookup.csv"

def read_tlc_trips(path):
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("cloudFiles.schemaHints", "customer_id BIGINT")
        .option("rescuedDataColumn", "_rescued_data")
        .option("pathGlobFilter", "*.parquet")
        .option("recursiveFileLookup", "true")
        .load(path)
    )


In [0]:
@dp.table(
    name="bronze.yellow_trips_raw",
    comment="Raw Yellow Taxi records",
    table_properties={"delta.feature.timestampNtz": "supported"}
)
def yellow_trips_raw():
    return prepare_bronze(read_tlc_trips(yellow_source_path), "tpep_pickup_datetime")


@dp.table(
    name="bronze.green_trips_raw",
    comment="Raw Green Taxi records",
    table_properties={"delta.feature.timestampNtz": "supported"}
)
def green_trips_raw():
    return prepare_bronze(read_tlc_trips(green_source_path), "lpep_pickup_datetime")


@dp.materialized_view(
    name="bronze.taxi_zones_raw",
    comment="Static NYC Taxi zone reference data"
)
def taxi_zones_raw():
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(zones_source_path)
    )